# Adım 5: Özellik Mühendisliği (Feature Engineering)
**Proje:** Spotify Büyük Veri Analizi  
**Veri Seti:** Spotify Tracks Dataset (~114K şarkı)  
Bu notebook'ta ML modeli için anlamlı 5 yeni özellik üretip Delta Lake Gold katmanına kaydedildi.

#### 1.Adım - Kütüphaneleri ve Spark Oturumunu Başlatma İşlemi Yapıldı

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, min as spark_min, max as spark_max, when, udf, avg, count
)
from pyspark.sql.types import StringType, IntegerType, DoubleType
import warnings
warnings.filterwarnings('ignore')  

# spark oturumu oluşturma delta lake ile
spark = SparkSession.builder \
    .appName("Spotify-FeatureEngineering") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# Log seviyesini WARN'a çekiyoruz - konsolu temiz tutar
#spark.sparkContext.setLogLevel("WARN")

print("Spark oturumu başarıyla başladı...")
print(f"Spark versiyonu: {spark.version}")

Spark oturumu başarıyla başladı...
Spark versiyonu: 3.5.0


#### 2.Adım Veriyi Delta Lake Silver Dosyasından Yükleme İşlemi

In [2]:
# deltalake silver dosyasından temizlenmiş verinin okunması ve kontrolu
SILVER_PATH = "/home/jovyan/delta-lake/silver"
GOLD_PATH   = "/home/jovyan/delta-lake/gold/features"

df = spark.read.format("delta").load(SILVER_PATH)
#databriks test için 
#df = spark.read.parquet("/Volumes/workspace/default/silver/")
print(f"Veri yüklendi...")
print(f"Toplam satır: {df.count():,}")
print(f"Toplam kolon: {len(df.columns)}")

Veri yüklendi...
Toplam satır: 89,740
Toplam kolon: 23


#### 3. Adım - özellik 1- energy_danceability_ratio
EDA Adım 9'da türe göre enerji ve dans edilebilirlik karşılaştırıldığında bazı türlerin yüksek enerjili ama düşük dans edilebilirliğe sahip olduğu gözlemlendi.Bu özellik o ayrımı sayısal olarak ifade etmek için üretildi.

In [3]:
# enerji/danceability : Değer yüksekse şarkı enerjik ama dans edilemez → Rock, metal gibi 'agresif/sert' müzik türlerini diğerlerinden ayırt etmek için kullanılır.
# Paydaya 0.0001 ekledik ki danceability 0 gelirse hata vermesin

df = df.withColumn("energy_danceability_ratio",col("energy")/(col("danceability") + 0.0001))

# hesaplanan degerlerin kontrolu yapildi mean/min/max seklinde 
stats = df.select(
    spark_min("energy_danceability_ratio").alias("minimum"),
    spark_max("energy_danceability_ratio").alias("maksimum"),
    avg("energy_danceability_ratio").alias("ortalama")
).collect()[0]

print("energy_danceability_ratio")
print(f"  Minimum  : {stats['minimum']:.4f}")
print(f"  Maksimum : {stats['maksimum']:.4f}")
print(f"  Ortalama : {stats['ortalama']:.4f}")

print("en yüksek energy_danceability_ratio")
df.orderBy(col("energy_danceability_ratio").desc()) \
  .select("track_name", "artists", "track_genre",
          "energy", "danceability", "energy_danceability_ratio") \
  .show(10, truncate=30)

print("en dusuk energy_danceability_ratio")
df.orderBy(col("energy_danceability_ratio").asc()) \
  .select("track_name", "artists", "track_genre",
          "energy", "danceability", "energy_danceability_ratio") \
  .show(10, truncate=30)

energy_danceability_ratio
  Minimum  : 0.0000
  Maksimum : 9990.0001
  Ortalama : 3.4189
en yüksek energy_danceability_ratio
+------------------------------+------------------------------+-----------+------+------------+-------------------------+
|                    track_name|                       artists|track_genre|energy|danceability|energy_danceability_ratio|
+------------------------------+------------------------------+-----------+------+------------+-------------------------+
|Hotel Hair Dryer - Non-Stat...|Deep Sleep Hair Dryers;Hair...|      sleep| 0.999|         0.0|        9990.000128746033|
|               Continuous Rain|Rain Sounds;Sounds Of Natur...|      sleep| 0.998|         0.0|        9980.000257492065|
|               Continuous Rain|Rain for Deep Sleep;Yoga;Th...|      sleep| 0.998|         0.0|        9980.000257492065|
|             Ankara Rain Skies|                Sound Sleeping|      sleep| 0.994|         0.0|        9940.000176429749|
|        The Early Mo

Çıkarım: En yüksek skorlar sleep türünden geliyor. Bu şarkılarda energy yüksek (beyaz gürültü, yağmur sesi gibi sürekli ses var) ama danceability sıfır. En düşük skorlarda ise energy neredeyse sıfır olan sessiz guitar, goth ve emo parçaları var. Özellik beklenen davranışı gösteriyor. Rock/metal yerine sleep öne çıkmasının nedeni Spotify'ın energy'yi agresiflik değil ses yoğunluğu olarak ölçmesi olabilir.

#### 4. Adım - ozellik 2- loudness_normalized
EDA Adım 3.1'de loudness'ın -49.5 ile 4.5 arasında değiştiği gözlemlenmişti.Negatif değerler ML modeline doğrudan verilmesi doğru olmadığı için 0-1 arasına normalize edilme işlemi yapıldı

In [4]:
# once ses yuksekliği normalizasyon islemi için  min ve max değerleri hesaplandı 
min_loudness = df.select(spark_min("loudness")).collect()[0][0]
max_loudness = df.select(spark_max("loudness")).collect()[0][0]

print(f"Loudness min: {min_loudness:.2f} dB")
print(f"Loudness max: {max_loudness:.2f} dB")

# normalize edilme islemi yapildi
df = df.withColumn(
    "loudness_normalized",
    (col("loudness") - min_loudness) / (max_loudness - min_loudness)
)

stats = df.select(
    spark_min("loudness_normalized").alias("minimum"),
    spark_max("loudness_normalized").alias("maksimum"),
    avg("loudness_normalized").alias("ortalama")
).collect()[0]

print("loudness normalize değerleri")
print(f"  Minimum  : {stats['minimum']:.4f}")
print(f"  Maksimum : {stats['maksimum']:.4f}")
print(f"  Ortalama : {stats['ortalama']:.4f}")

# orjinal ve yeni normalize degerlerin karsilastirmasi
print("\nOrijinal vs Normalize karşılaştırma (5 satır):")
df.select("track_name", "loudness", "loudness_normalized").show(5, truncate=25)

Loudness min: -49.53 dB
Loudness max: 4.53 dB
loudness normalize değerleri
  Minimum  : 0.0000
  Maksimum : 1.0000
  Ortalama : 0.7590

Orijinal vs Normalize karşılaştırma (5 satır):
+-------------------------+--------+-------------------+
|               track_name|loudness|loudness_normalized|
+-------------------------+--------+-------------------+
|Böxig Leise - Pig & Da...| -13.264| 0.6708284674341414|
|               Teeje Week|  -3.537| 0.8507481963594011|
|                 Poor Man| -11.942| 0.6952814129155639|
|          Love Generation|  -9.744| 0.7359376847382425|
|                Addiction|  -0.647| 0.9042043496850037|
+-------------------------+--------+-------------------+
only showing top 5 rows



Çıkarım: loudness değişkeni -49.53 ile 4.53 dB arasında değişmektedir. Negatif değerlerin ML modeline doğrudan verilmesi uygun olmadığından min-max normalizasyonu uygulanarak 0-1 arasına ölçeklendirilmiştir. Normalize sonrası ortalama 0.7590 olup şarkıların büyük çoğunluğunun yüksek ses seviyesine sahip olduğu görülmektedir. Bu özellik tür tahmini modelinde ses yüksekliğini anlamlı bir girdi olarak sunmaktadır.

#### 5. adım - ozellik 3- tempo_category
EDA Adım 12'de yavas/orta/hizli gruplarının şarkı sayısı ve popülerlik dağılımı incelenmişti.tempo sayısal değerini kategoriye çeviriyoru
Sürekli bir sayı yerine anlamlı gruplar oluşturmak bazı modellerin örüntüleri daha iyi öğrenmesini saglayabilir. Orijinal tempo kolonuna ek olarak tempo sayısal degerini 3 kategori olarak gruplanması planlandı mle daha yorumlanabilir bilgi olarak sunmak icin

In [5]:
# tempo sayısal değerini 3 kategoriye çeviriyoruz
df = df.withColumn(
    "tempo_category",
    when(col("tempo") < 100, "yavas")       # 100 BPM'den yavaş
    .when((col("tempo") >= 100) & (col("tempo") <= 140), "orta")  # 100-140 BPM arası
    .otherwise("hizli")                     # 140 BPM'den hızlı
)

print("tempo_category")
# her kategoride kaç şarkı ve ortalama popülerlik var
df.groupBy("tempo_category") \
  .agg(
      count("*").alias("sarki_sayisi"),
      avg("tempo").alias("ort_tempo"),
      avg("popularity").alias("ort_populerlik")
  ) \
  .orderBy("ort_tempo") \
  .show()

print("Tempo ve kategori (30 örnek):")
df.select("track_name", "track_genre", "tempo", "tempo_category").show(30, truncate=30)

tempo_category
+--------------+------------+------------------+------------------+
|tempo_category|sarki_sayisi|         ort_tempo|    ort_populerlik|
+--------------+------------+------------------+------------------+
|         yavas|       23488| 85.50079007215331| 33.01881811989101|
|          orta|       43100|120.95551330194672|33.000533642691416|
|         hizli|       23152|161.19867761528894| 33.75051831375259|
+--------------+------------+------------------+------------------+

Tempo ve kategori (30 örnek):
+------------------------------+--------------+-------+--------------+
|                    track_name|   track_genre|  tempo|tempo_category|
+------------------------------+--------------+-------+--------------+
| Böxig Leise - Pig & Dan Remix|minimal-techno|119.997|          orta|
|                    Teeje Week|       hip-hop|161.721|         hizli|
|                      Poor Man|     bluegrass| 91.321|         yavas|
|               Love Generation|         happy|159.9

Çıkarım:tempo_category dağılımına bakıldığında şarkıların yarısından fazlası orta tempo grubunda (100-140 BPM) yer alıyor. Üç grubun ortalama popülerliği birbirine çok yakın (33.0-33.7) olup tempo'nun tek başına popülerliği belirlemediği görülüyor. Tür bazında incelendiğinde aynı türün farklı tempo kategorilerinde çıkabildiği gözlemlendi; örneğin black-metal orta, grunge hızlı, study ise yavaş kategorisinde yer aldı. Bu durum tempo_category'nin tek başına tür tahmini için yeterli olmadığını ancak loudness_normalized ve energy_acoustic_contrast gibi diğer özelliklerle birlikte modele anlamlı katkı sağlayabileceğini göstermektedir.

#### 6.Adım - ozellik 4 -energy_acoustic_contrast
EDA Adım 7 korelasyon heatmap'te acousticness ile energy arasında -0.72 korelasyon gözlemlenmişti. Pozitif değer yüksek enerjili ama düşük akustik türleri (elektronik, rock, metal) temsil ederken düşük değer akustik müzikleri öne çıkarır. Tür tahminine doğrudan katkı sağlayan güçlü bir ayırt edici özellik olarak ML modeline verilebilir.

In [6]:
# energy_acoustic_contrast = energy - acousticness
df = df.withColumn(
    "energy_acoustic_contrast",
    col("energy") - col("acousticness")
)

stats = df.select(
    spark_min("energy_acoustic_contrast").alias("minimum"),
    spark_max("energy_acoustic_contrast").alias("maksimum"),
    avg("energy_acoustic_contrast").alias("ortalama")
).collect()[0]

print("energy_acoustic_contrast")
print(f"  Minimum  : {stats['minimum']:.4f}")
print(f"  Maksimum : {stats['maksimum']:.4f}")
print(f"  Ortalama : {stats['ortalama']:.4f}")

# doğrulama
print("\nEn yüksek skorlar (akustik baskın):")
df.orderBy(col("energy_acoustic_contrast").desc()) \
  .select("track_name", "track_genre", "acousticness", "energy", "energy_acoustic_contrast") \
  .show(5, truncate=30)

print("\nEn düşük skorlar (sert/yoğun türler):")
df.orderBy(col("energy_acoustic_contrast").asc()) \
  .select("track_name", "track_genre", "acousticness", "energy", "energy_acoustic_contrast") \
  .show(5, truncate=30)

energy_acoustic_contrast
  Minimum  : -0.9959
  Maksimum : 0.9994
  Ortalama : 0.3062

En yüksek skorlar (akustik baskın):
+------------------------------+-----------+------------+------+------------------------+
|                    track_name|track_genre|acousticness|energy|energy_acoustic_contrast|
+------------------------------+-----------+------------+------+------------------------+
|                 Affront Final|black-metal|      6.5E-4|   1.0|                 0.99935|
|                         ObZen|death-metal|     3.17E-6| 0.999|              0.99899685|
|         Undertaker, Undertake|  power-pop|     4.86E-6| 0.999|               0.9989951|
| Cure for the Common Complaint|  grindcore|     5.27E-6| 0.999|              0.99899477|
|Kill Theme for American Ape...|  grindcore|     7.94E-6| 0.999|               0.9989921|
+------------------------------+-----------+------------+------+------------------------+
only showing top 5 rows


En düşük skorlar (sert/yoğun türler):
+--

energy_acoustic_contrast değerleri -0.9994 ile 0.9959 arasında değişmektedir. En yüksek skorlarda sleep, piano ve classical türleri öne çıkarken en düşük skorlarda black-metal, death-metal ve grindcore yer aldı. Ortalama -0.3062 olup veri setinde yüksek enerjili şarkıların daha fazla olduğu görülmektedir. Bu özellik akustik ve sert türler arasındaki farkı başarıyla yakalamakta olup tür tahmini modelinde güçlü bir ayırt edici özellik olarak kullanılabilir.

#### 7. Adım- ozellik 5- dancefloor_score
Dancefloor_score, danceability ve valence değerlerini birleştirerek hem dans edilebilir hem de pozitif duygu tonuna sahip şarkıları yakalar. Değer yüksekse şarkı dans pistine daha uygun ve daha neşeli kabul edilir.

In [7]:
df = df.withColumn(
    "dancefloor_score",
    (col("danceability") + col("valence")) / 2
)

stats = df.select(
    spark_min("dancefloor_score").alias("minimum"),
    spark_max("dancefloor_score").alias("maksimum"),
    avg("dancefloor_score").alias("ortalama")
).collect()[0]

print("dancefloor_score")
print(f"  Minimum  : {stats['minimum']:.4f}")
print(f"  Maksimum : {stats['maksimum']:.4f}")
print(f"  Ortalama : {stats['ortalama']:.4f}")

# doğrulama
print("\nEn yüksek dancefloor_score (dans pisti türleri):")
df.orderBy(col("dancefloor_score").desc()) \
  .select("track_name", "track_genre", "danceability", "valence", "dancefloor_score") \
  .show(10, truncate=30)

print("\nEn düşük dancefloor_score:")
df.orderBy(col("dancefloor_score").asc()) \
  .select("track_name", "track_genre", "danceability", "valence", "dancefloor_score") \
  .show(10, truncate=30)

dancefloor_score
  Minimum  : 0.0000
  Maksimum : 0.9790
  Ortalama : 0.5158

En yüksek dancefloor_score (dans pisti türleri):
+-----------------------------+-----------+------------+-------+------------------+
|                   track_name|track_genre|danceability|valence|  dancefloor_score|
+-----------------------------+-----------+------------+-------+------------------+
|               Hot Cross Buns|   children|       0.976|  0.982|0.9789999723434448|
|               I Like Pencils|       kids|       0.962|  0.992|0.9769999980926514|
|      Polly Put the Kettle On|   children|       0.973|  0.973|0.9729999899864197|
|            Itsy Bitsy Spider|   children|       0.952|  0.983|0.9674999713897705|
|                Letter Y Song|       kids|       0.968|  0.962|0.9650000333786011|
|                 Akkad Bakkad|   children|       0.961|  0.968|0.9645000100135803|
|   Five Little Speckled Frogs|       kids|       0.962|  0.966|0.9639999866485596|
|London Bridge Is Falling Down|  

dancefloor_score, danceability ve valence ortalamasıyla hesaplanır. En yüksek değerler hem dans edilebilirliği hem de pozitif duygu tonu yüksek şarkıları, düşük değerler ise dans edilebilirliği veya valence değeri düşük şarkıları gösterir.

####  özelliklerin veriseti üzerinde kontrol edilmesi

In [8]:
yeni_ozellikler = [
    "energy_danceability_ratio",
    "loudness_normalized",
    "tempo_category",
    "energy_acoustic_contrast",
    "dancefloor_score",
]

print("Yeni Özellikler (15 örnek):")
display(df.select(["track_name", "track_genre"] + yeni_ozellikler).limit(15).toPandas())

sayisal_yeni = [o for o in yeni_ozellikler if o != "tempo_category"]
print("\nİstatistikler:")
display(df.select(sayisal_yeni).describe().toPandas())

print("\nTempo Kategori Dağılımı:")
display(df.groupBy("tempo_category").count().orderBy("tempo_category").toPandas())

Yeni Özellikler (15 örnek):


,track_name,track_genre,energy_danceability_ratio,loudness_normalized,tempo_category,energy_acoustic_contrast,dancefloor_score
0,Böxig Leise - Pig & Dan Remix,minimal-techno,0.816208,0.670828,orta,0.558860,0.3970
1,Teeje Week,hip-hop,1.133854,0.850748,hizli,0.711700,0.7590
2,Poor Man,bluegrass,0.499914,0.695281,yavas,0.029000,0.5385
3,Love Generation,happy,1.788740,0.735938,hizli,0.948780,0.5420
4,Addiction,idm,1.768216,0.904204,hizli,0.897000,0.4295
5,Mr. Brightside,alt-rock,2.587333,0.819433,hizli,0.909790,0.2940
6,Lovemark,emo,0.409603,0.667259,hizli,0.200000,0.5725
7,Got You On My Mind,french,0.653482,0.790430,orta,0.169000,0.6480
8,Agora Estou Sofrendo - Ao Vivo,forro,1.120476,0.855927,orta,0.036000,0.5220
9,All I Know - M&F's Rolling Out Radio Mix,drum-and-bass,1.833300,0.871039,hizli,0.654000,0.6550



İstatistikler:


,summary,energy_danceability_ratio,loudness_normalized,energy_acoustic_contrast,dancefloor_score
0,count,89740,89740,89740,89740
1,mean,3.4189033598584255,0.7589664976861054,0.3061735018714145,0.5158203963151295
2,stddev,116.26634000994305,0.09658209491822747,0.5545303379874025,0.19109796611849084
3,min,0.0,0.0,-0.995941,0.0
4,max,9990.000128746033,1.0,0.99935,0.9789999723434448



Tempo Kategori Dağılımı:


,tempo_category,count
0,hizli,23152
1,orta,43100
2,yavas,23488


##### ozellik tablosunu deltalake gold dosyasına kaydetme islemi 

In [9]:
# ml için golda kaydedilecek attributelar
gold_kolonlar = [
    # benzersiz ozellikler
    "track_id","track_name", "artists", "album_name", 
    #hedef
    "track_genre","popularity",
    # orijinal sayısal veriler
    "danceability", "energy", "loudness", "tempo",
    "speechiness", "acousticness", "instrumentalness", "liveness",
    "valence", "duration_ms", "key", "mode", "time_signature",
    # orijinal kategorik ozellik
    "explicit",
    # kafka timestamp
    "kafka_timestamp",'user_id', 'event_type',
    # uretilen +5 ozellik
    "energy_danceability_ratio",  "loudness_normalized", "tempo_category","energy_acoustic_contrast","dancefloor_score",           
]
mevcut_kolonlar = [c for c in gold_kolonlar if c in df.columns]
df_gold = df.select(mevcut_kolonlar)
print(f"df'deki toplam kolon sayısı   : {len(df.columns)}")
print(f"Gold'a yazılacak kolon sayısı : {len(mevcut_kolonlar)}")
print(f"Gold'a yazılacak satır sayısı : {df_gold.count():,}")
print(f"\nEksik kolonlar (df'de yok)    : {[c for c in gold_kolonlar if c not in df.columns]}")

df'deki toplam kolon sayısı   : 28
Gold'a yazılacak kolon sayısı : 28
Gold'a yazılacak satır sayısı : 89,740

Eksik kolonlar (df'de yok)    : []


In [10]:
#yazdırma islemi
GOLD_PATH = "/home/jovyan/delta-lake/gold/features"

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(GOLD_PATH)

print(f"Özellik tablosu Gold katmanına kaydedildi")
print(f"Konum: {GOLD_PATH}")

Özellik tablosu Gold katmanına kaydedildi
Konum: /home/jovyan/delta-lake/gold/features


In [11]:
# Kaydın başarılı olduğunu dogrulama işlemi
df_verify = spark.read.format("delta").load(GOLD_PATH)
print("Gold Katmanından Okunan Veri")
print(f"Satır sayısı   : {df_verify.count():,}")
print(f"Kolon sayısı   : {len(df_verify.columns)}")
print("\nKolon listesi:")
for kolon in df_verify.columns:
    print(f"{kolon}")
print("\nİlk 5 satır (yeni özellikler):")
display(df_verify.select(
    "track_genre",
    "energy_danceability_ratio",
    "loudness_normalized",
    "tempo_category",
    "energy_acoustic_contrast",
    "dancefloor_score"
).limit(5).toPandas())
print("\nİlk 5 satır:")

display(df_verify.limit(5).toPandas())

Gold Katmanından Okunan Veri
Satır sayısı   : 89,740
Kolon sayısı   : 28

Kolon listesi:
track_id
track_name
artists
album_name
track_genre
popularity
danceability
energy
loudness
tempo
speechiness
acousticness
instrumentalness
liveness
valence
duration_ms
key
mode
time_signature
explicit
kafka_timestamp
user_id
event_type
energy_danceability_ratio
loudness_normalized
tempo_category
energy_acoustic_contrast
dancefloor_score

İlk 5 satır (yeni özellikler):


,track_genre,energy_danceability_ratio,loudness_normalized,tempo_category,energy_acoustic_contrast,dancefloor_score
0,chill,0.768227,0.793278,hizli,0.15500,0.50950
1,drum-and-bass,2.139244,0.869116,hizli,0.88290,0.28050
2,punk-rock,1.981684,0.766291,yavas,0.77599,0.45900
3,honky-tonk,0.335371,0.726430,orta,-0.61100,0.52150
4,world-music,0.058199,0.248765,hizli,-0.98596,0.11225



İlk 5 satır:


,track_id,track_name,artists,album_name,track_genre,popularity,danceability,energy,loudness,tempo,...,time_signature,explicit,kafka_timestamp,user_id,event_type,energy_danceability_ratio,loudness_normalized,tempo_category,energy_acoustic_contrast,dancefloor_score
0,001APMDOl3qtx1526T11n1,Better,Pink Sweat$;Kirby,New RnB,chill,0,0.6130,0.47100,-6.644000,143.063995,...,4,0,2026-05-12T08:30:09.485503,b52bb185-7c53-498a-bf86-21ea6bac6653,track_played,0.768227,0.793278,hizli,0.15500,0.50950
1,002uYDBLOvJz21C4FuArDS,Find Me - Sigma VIP Remix,Sigma;Birdy,Find Me (Remixes),drum-and-bass,20,0.4150,0.88800,-2.544000,174.985992,...,4,0,2026-05-12T08:30:48.163904,c9e5ebcc-93cf-423b-9135-fd446b14eb09,track_played,2.139244,0.869116,hizli,0.88290,0.28050
2,004G9E3EZhxxn5aE9yEQqx,Sandwiches de Miga,Pappo's Blues,"Pappo's Blues, Vol. 3",punk-rock,36,0.3930,0.77900,-8.103000,88.418999,...,4,0,2026-05-12T08:33:31.483209,43ad7111-d8e9-46db-a88b-7c46f84fd250,track_played,1.981684,0.766291,yavas,0.77599,0.45900
3,004iWPkSRbvOEvAPLWHl9M,Mister Love,Ernest Tubb;The Wilburn Brothers,Definitive Hits,honky-tonk,13,0.6380,0.21400,-10.258000,117.764999,...,4,0,2026-05-12T08:31:58.118275,5e201e18-74d2-488a-bca0-969be96a4313,track_played,0.335371,0.726430,orta,-0.61100,0.52150
4,006ATYzgynEKIPgVaT5LQM,528Hz Energía curativa profunda,Mc_team,Frecuencias Curativas Solfeggio 528 Hz,world-music,24,0.0865,0.00504,-36.082001,169.362000,...,4,0,2026-05-12T08:34:49.268443,e49868a0-e44b-4f81-9b80-2fe1862e510b,track_played,0.058199,0.248765,hizli,-0.98596,0.11225
